# Member 2: Candidate generation and feature engineering

This notebook demonstrates only Member 2's work on one trip: build metric road segments, generate nearby arc candidates, calculate candidate features, align route labels, and report candidate recall.

The route-to-GPS alignment is a monotonic nearest-route heuristic because the route file does not label individual GPS timestamps directly. Candidate recall here measures coverage against that heuristic.

In [ ]:
from pathlib import Path
import io
import sys
import zipfile
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "features").is_dir() and (REPO_ROOT.parent / "src" / "features").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.features import build_candidate_table
from src.features.create_training_data import align_reference_route, candidate_recall_report
from src.features.road_network import build_road_network

DATA_ROOT = Path.home() / "Downloads" / "Student folder"
PARTITION = "train"
TRIP_ID = "00000001"
CANDIDATE_RADIUS_M = 100.0
TRIP_FOLDER = DATA_ROOT / PARTITION / TRIP_ID
print("Repository:", REPO_ROOT)
print("Trip folder:", TRIP_FOLDER)

In [ ]:
if TRIP_FOLDER.is_dir():
    try:
        from src.preprocessing.data_loader import load_trip
        trip = load_trip(TRIP_FOLDER)
    except ModuleNotFoundError:
        trip = {
            "trip_id": TRIP_ID,
            "track": pd.read_csv(TRIP_FOLDER / f"{TRIP_ID}.track", sep=r"\s+", header=None,
                                  names=["longitude", "latitude", "timestamp"]),
            "nodes": pd.read_csv(TRIP_FOLDER / f"{TRIP_ID}.nodes", sep=r"\s+", header=None,
                                 names=["longitude", "latitude"]),
            "arcs": pd.read_csv(TRIP_FOLDER / f"{TRIP_ID}.arcs", sep=r"\s+", header=None,
                                names=["from_node", "to_node"]),
            "route": pd.read_csv(TRIP_FOLDER / f"{TRIP_ID}.route", sep=r"\s+", header=None,
                                 names=["arc_id"]),
        }
else:
    archive_options = [
        Path.home() / "Downloads" / "PathSnap_student_materials.zip",
        Path.home() / ".codex" / ".chatgpt-projects" / "g-p-6afd3005b7248191a0a14faae7c74479"
        / "sources" / "PathSnap_student_materials.zip",
    ]
    archive_path = next((path for path in archive_options if path.is_file()), None)
    if archive_path is None:
        raise FileNotFoundError(
            "No extracted trip folder or PathSnap_student_materials.zip found. "
            "Set DATA_ROOT or add the archive path to archive_options."
        )
    outer = zipfile.ZipFile(archive_path)
    inner = zipfile.ZipFile(io.BytesIO(outer.read("Student folder.zip")))
    prefix = f"Student folder/{PARTITION}/{TRIP_ID}/{TRIP_ID}"
    read_member = lambda suffix, names: pd.read_csv(
        io.BytesIO(inner.read(prefix + suffix)), sep=r"\s+", header=None, names=names
    )
    trip = {
        "trip_id": TRIP_ID,
        "track": read_member(".track", ["longitude", "latitude", "timestamp"]),
        "nodes": read_member(".nodes", ["longitude", "latitude"]),
        "arcs": read_member(".arcs", ["from_node", "to_node"]),
        "route": read_member(".route", ["arc_id"]),
    }
    print("Loaded from archive:", archive_path)

print("Trip:", trip["trip_id"])
print("Loader keys:", list(trip))

## 1. Inspect this trip's inputs

These counts confirm the loader data passed to Member 2. This is a focused pipeline demo, not Member 1's full dataset EDA.

In [ ]:
summary = pd.DataFrame([
    {"file": ".track", "rows": len(trip["track"]), "meaning": "GPS observations"},
    {"file": ".nodes", "rows": len(trip["nodes"]), "meaning": "road node coordinates"},
    {"file": ".arcs", "rows": len(trip["arcs"]), "meaning": "directed road connections"},
    {"file": ".route", "rows": len(trip["route"]), "meaning": "ordered reference arc IDs"},
])
display(summary)


## 2. Build the road graph and candidate feature table

Coordinates are projected to local metres before distances and bearings are calculated. Each candidate row represents one possible road arc for one GPS observation.

In [ ]:
nodes = list(trip["nodes"][["longitude", "latitude"]].itertuples(index=False, name=None))
arcs = list(trip["arcs"][["from_node", "to_node"]].itertuples(index=False, name=None))
track = list(trip["track"][["longitude", "latitude", "timestamp"]].itertuples(index=False, name=None))
route_arc_ids = trip["route"]["arc_id"].astype(int).tolist()

network = build_road_network(nodes, arcs)
points_xy = [network.projection.project(lon, lat) for lon, lat, _ in track]
expected_arc_ids = align_reference_route(points_xy, route_arc_ids, network)
candidate_rows = build_candidate_table(
    trip,
    candidate_radius_m=CANDIDATE_RADIUS_M,
    include_labels=True,
)
candidate_table = pd.DataFrame(candidate_rows)

print("Candidate rows:", len(candidate_table))
print("Candidate radius (m):", CANDIDATE_RADIUS_M)
display(candidate_table.head(10))


## 3. Check candidate recall

Candidate recall asks: for how many observations did candidate generation include the arc selected by the route-alignment rule? It evaluates candidate coverage before the ML model is involved.

In [ ]:
recall = candidate_recall_report(candidate_rows, expected_arc_ids, trip["trip_id"])
display(pd.Series(recall, name="value"))

# Show candidate counts and the best-ranked candidates for the first few observations.
per_observation = candidate_table.groupby("observation_idx").size().rename("candidate_count")
display(per_observation.head(10).to_frame())
display(candidate_table[candidate_table["observation_idx"] == 0].head(10))


## What to explain in the demo

- `.arcs` row position is the arc ID; `.nodes` row position is the node ID.
- Candidate generation keeps arcs within the configured radius and ranks them by distance.
- Features include GPS-to-arc distance, projected closest point, along-arc fraction, speed, and bearing difference.
- The reference route is used only when `include_labels=True`; inference should leave it disabled.
- Alignment preserves route order but is still a heuristic. Report candidate recall as coverage against that heuristic, not as final model accuracy.

To explore another training trip, change `TRIP_ID` near the top and rerun the notebook. Use only training and student-development trips to tune the candidate radius.